# 0.0 Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.tree import DecisionTreeRegressor
from sklearn import metrics as mt

# 0.1 - Load Datasets

In [2]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [3]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [4]:
# Instanciar o modelo com parâmetros default
model_def = DecisionTreeRegressor(random_state=42)

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [5]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.9918
MSE:  3.94
RMSE: 1.99
MAE:  0.21
MAPE: 8.26%


## Passo 3 — Performance na validação (default)

In [6]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   -0.3232
MSE:  631.85
RMSE: 25.14
MAE:  17.33
MAPE: 724.98%


## Passo 4 — Ajuste de hiperparâmetros

In [7]:
print("--- Testando Múltiplos Hiperparâmetros na Árvore de Decisão (Regressão) ---")
melhor_max_depth = None
melhor_min_samples_split = 2
melhor_r2_val = -999

# Lista de Parametros para cruzar no loop
list_max_depth = [2, 5, 10, 15, 20, None]
list_min_samples_split = [2, 5, 10, 20, 50]

# LOOP DUPLO: Cruza Max_Depth x Min_Samples_Split
for md in list_max_depth:
    for mss in list_min_samples_split:
        
        # 1. Instanciar o Regressor de Árvore
        model = DecisionTreeRegressor(
            max_depth=md, 
            min_samples_split=mss, 
            random_state=42
        )
        
        try:
            # 2. Treinamento
            model.fit(X_train, y_train.values.ravel())
            
            # 3. Predict
            yhat_train = model.predict(X_train)
            yhat_val = model.predict(X_val)
            
            # 4. Cálculo das Métricas
            r2_train = mt.r2_score(y_train, yhat_train)
            r2_val = mt.r2_score(y_val, yhat_val)
            rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
            mae_val = mt.mean_absolute_error(y_val, yhat_val)
            
            # Tratamento visual do MAPE para evitar quebras
            mape_raw = mt.mean_absolute_percentage_error(y_val, yhat_val)
            mape_print = f"{mape_raw * 100:.2f}%" if np.isfinite(mape_raw) and mape_raw < 1000 else "Distort (High)"
            
            # Formatação do print tratando o 'None' como string
            md_print = "None" if md is None else f"{md:4d}"
            print(f"Max_Depth: {md_print:<4} | Min_Split: {mss:2d} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f} | MAPE Val: {mape_print}")
            
            # Guardar a melhor combinação baseada no R² de validação
            if r2_val > melhor_r2_val:
                melhor_r2_val = r2_val
                melhor_max_depth = md
                melhor_min_samples_split = mss
                
        except Exception as e:
            print(f"Erro na combinação Max_Depth {md} | Min_Split {mss}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO DECISION TREE:")
print(f"Melhor Max_Depth: {melhor_max_depth}")
print(f"Melhor Min_Samples_Split: {melhor_min_samples_split}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Testando Múltiplos Hiperparâmetros na Árvore de Decisão (Regressão) ---
Max_Depth:    2 | Min_Split:  2 | R² Treino: 0.0433 | R² Val: 0.0376 | RMSE Val: 21.44 | MAPE Val: 848.01%
Max_Depth:    2 | Min_Split:  5 | R² Treino: 0.0433 | R² Val: 0.0376 | RMSE Val: 21.44 | MAPE Val: 848.01%
Max_Depth:    2 | Min_Split: 10 | R² Treino: 0.0433 | R² Val: 0.0376 | RMSE Val: 21.44 | MAPE Val: 848.01%
Max_Depth:    2 | Min_Split: 20 | R² Treino: 0.0433 | R² Val: 0.0376 | RMSE Val: 21.44 | MAPE Val: 848.01%
Max_Depth:    2 | Min_Split: 50 | R² Treino: 0.0433 | R² Val: 0.0376 | RMSE Val: 21.44 | MAPE Val: 848.01%
Max_Depth:    5 | Min_Split:  2 | R² Treino: 0.1135 | R² Val: 0.0636 | RMSE Val: 21.15 | MAPE Val: 839.58%
Max_Depth:    5 | Min_Split:  5 | R² Treino: 0.1135 | R² Val: 0.0636 | RMSE Val: 21.15 | MAPE Val: 839.58%
Max_Depth:    5 | Min_Split: 10 | R² Treino: 0.1135 | R² Val: 0.0636 | RMSE Val: 21.15 | MAPE Val: 839.58%
Max_Depth:    5 | Min_Split: 20 | R² Treino: 0.1135 | R² Val: 0.0636

## Passo 5 — Performance com modelo tunado (treino e validação)

In [8]:
model_tunado = DecisionTreeRegressor(
    max_depth=melhor_max_depth,
    min_samples_split=melhor_min_samples_split,
    random_state=42
)
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"MÉTRICAS TREINAMENTO (Tunado):")
print(f"R² (Coef. Determinação): {r2_train_tunado:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_train_tunado:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_train_tunado:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_train_tunado:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_train_tunado * 100:.2f}%")
print("=" * 65)
print(f"MÉTRICAS VALIDAÇÃO (Tunado):")
print(f"R² (Coef. Determinação): {r2_val_tunado:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_val_tunado:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_val_tunado:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_val_tunado:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_val_tunado * 100:.2f}%")

--- Performance com Modelo Tunado ---
MÉTRICAS TREINAMENTO (Tunado):
R² (Coef. Determinação): 0.1135
MSE (Erro Quadrático Médio): 423.75
RMSE (Raiz do Erro Quadrático): 20.59
MAE (Erro Absoluto Médio): 16.37
MAPE (Erro Percentual Médio): 786.95%
MÉTRICAS VALIDAÇÃO (Tunado):
R² (Coef. Determinação): 0.0636
MSE (Erro Quadrático Médio): 447.16
RMSE (Raiz do Erro Quadrático): 21.15
MAE (Erro Absoluto Médio): 16.84
MAPE (Erro Percentual Médio): 839.58%


## Passo 6 — União treino + validação

In [9]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [10]:
model_final = DecisionTreeRegressor(
    max_depth=melhor_max_depth,
    min_samples_split=melhor_min_samples_split,
    random_state=42
)
model_final.fit(X_train_final, y_train_final.values.ravel())

print(f"Modelo final retreinado com max_depth={melhor_max_depth}, min_samples_split={melhor_min_samples_split}")
print(f"Amostras de treinamento final: {X_train_final.shape[0]}")

Modelo final retreinado com max_depth=5, min_samples_split=2
Amostras de treinamento final: 15068


## Passo 8 — Performance no teste

In [11]:
print("--- Executando o Teste Final ---")

yhat_test = model_final.predict(X_test)

r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print(f"MÉTRICAS NO DATASET DE TESTE:")
print(f"R² (Coef. Determinação): {r2_test:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_test:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_test:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_test:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_test * 100:.2f}%")
print("=" * 65)

--- Executando o Teste Final ---
MÉTRICAS NO DATASET DE TESTE:
R² (Coef. Determinação): 0.0896
MSE (Erro Quadrático Médio): 443.30
RMSE (Raiz do Erro Quadrático): 21.05
MAE (Erro Absoluto Médio): 16.83
MAPE (Erro Percentual Médio): 788.62%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [12]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%',   f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto        R²      RMSE       MAE    MAPE
   Treino (Default)  0.991757  1.985045  0.214099   8.26%
Validação (Default) -0.323224 25.136726 17.328398 724.98%
    Treino (Tunado)  0.113523 20.585122 16.368766 786.95%
 Validação (Tunado)  0.063559 21.146189 16.843452 839.58%
      Teste (Final)  0.089557 21.054587 16.829602 788.62%
